In [10]:
# bo_entrypoint_growing_multid_tuned.py
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from scipy.stats import norm
from scipy.optimize import minimize

try:
    from scipy.stats import qmc
    HAVE_QMC = True
except Exception:
    HAVE_QMC = False


def propose_next_query_point_growing_Dd_tuned(
    D,
    X_PATH,
    Y_PATH,
    SAVE_PATH,
    MINIMISE=True,
    GRID_CANDIDATES=30000,
    N_RESTARTS_ACQ=25,
    XI=None,
    RANDOM_SEED=0,
    BOUNDS=None,          # if known: list of (low, high) tuples of length D
    USE_SOBOL=True,
):
    """
    Hyper-tuned Bayesian Optimisation entry point for a chosen dimension D.

    - Loads X and y
    - Forces X to NxD safely (takes first D columns if X has more than D)
    - Fits a GP surrogate (hyperparams tuned via LML with restarts)
    - Proposes next query via Expected Improvement (EI)
    - Saves x_next to SAVE_PATH

    Works for D = 2, 3, 4, 5, 6, 8 (or any D >= 1).
    """
    rng = np.random.default_rng(RANDOM_SEED)

    # ---- Load data ----
    X = np.load(X_PATH)
    y = np.load(Y_PATH)

    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    # Ensure X is 2D matrix
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    elif X.ndim > 2:
        X = X.reshape(-1, X.shape[-1])

    if X.shape[1] < D:
        raise ValueError(f"Need at least D={D} columns, got X.shape={X.shape}")

    Xd = X[:, :D]  # safe: take first D dims

    if Xd.shape[0] != y.shape[0]:
        raise ValueError(f"Row mismatch: X has {Xd.shape[0]} rows but y has {y.shape[0]} values.")

    n, d = Xd.shape  # d == D

    # ---- XI schedule (exploration strength) ----
    if XI is None:
        if n < 15:
            XI = 1e-2
        elif n < 40:
            XI = 5e-3
        else:
            XI = 1e-3

    # ---- Bounds ----
    if BOUNDS is not None:
        if len(BOUNDS) != d:
            raise ValueError(f"BOUNDS length {len(BOUNDS)} does not match d={d}.")
        bounds = [(float(a), float(b)) for (a, b) in BOUNDS]
    else:
        mins = Xd.min(axis=0)
        maxs = Xd.max(axis=0)
        margin = 1e-6
        bounds = [(float(mins[i] - margin), float(maxs[i] + margin)) for i in range(d)]

    lb = np.array([b[0] for b in bounds], dtype=float)
    ub = np.array([b[1] for b in bounds], dtype=float)

    # ---- Fit GP (hyperparameters tuned via LML) ----
    # More restarts helps in low-data regimes; reduce as n grows.
    gp_restarts = 10 if n <= 30 else 5

    kernel = (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e1))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=gp_restarts
    )
    gp.fit(Xd, y)

    y_best = float(np.min(y)) if MINIMISE else float(np.max(y))

    # ---- Expected Improvement ----
    def expected_improvement(x):
        x = np.asarray(x).reshape(1, d)
        mu, sigma = gp.predict(x, return_std=True)
        mu = float(mu.ravel()[0])
        sigma = max(float(sigma.ravel()[0]), 1e-12)

        # IMPORTANT FIX: do NOT assign to y_best here
        imp = (y_best - mu - XI) if MINIMISE else (mu - y_best - XI)
        Z = imp / sigma
        return float(imp * norm.cdf(Z) + sigma * norm.pdf(Z))

    # ---- Candidate generation ----
    if USE_SOBOL and HAVE_QMC:
        # Sobol best with 2^m points; overshoot then slice
        m = int(np.ceil(np.log2(GRID_CANDIDATES)))
        sampler = qmc.Sobol(d=d, scramble=True, seed=RANDOM_SEED)
        cand01 = sampler.random_base2(m=m)[:GRID_CANDIDATES]
        candidates = lb + (ub - lb) * cand01
    else:
        candidates = rng.uniform(lb, ub, size=(int(GRID_CANDIDATES), d))

    eis = np.array([expected_improvement(c) for c in candidates], dtype=float)
    finite = np.isfinite(eis)
    if not np.any(finite):
        raise ValueError("All EI values are non-finite. Check bounds/model.")

    x0 = candidates[np.nonzero(finite)[0][np.argmax(eis[finite])]]

    # ---- Local refinement ----
    def neg_ei(x):
        return -expected_improvement(x)

    best = x0.copy()
    best_val = -neg_ei(x0)

    for _ in range(int(N_RESTARTS_ACQ)):
        x_init = rng.uniform(lb, ub, size=(d,))
        res = minimize(neg_ei, x_init, bounds=bounds, method="L-BFGS-B")
        if res.success and -res.fun > best_val:
            best_val = -res.fun
            best = res.x.copy()

    # Fallback: EI flat -> max uncertainty
    if best_val <= 1e-12:
        _, stds = gp.predict(candidates, return_std=True)
        best = candidates[np.argmax(stds)]

    np.save(SAVE_PATH, best)

    print(f"=== BO Next Query (GROWING {d}D tuned) ===")
    print("X_PATH:", X_PATH)
    print("Y_PATH:", Y_PATH)
    print("n points:", n)
    print("Chosen XI:", XI)
    print("Proposed next query point:", best)
    print("Saved to:", SAVE_PATH)
    print("Current best y:", y_best)
    print("Bounds:", bounds)
    print("GP kernel (fit):", gp.kernel_)

    return best


if __name__ == "__main__":
    # EDIT THESE PATHS per your files:
    baseX = "initial_inputs.npy"
    baseY = "initial_outputs.npy"

    # Example runs (NOW includes 2D):
   # propose_next_query_point_growing_Dd_tuned(
      #  D=2, X_PATH=baseX, Y_PATH=baseY,
      #  SAVE_PATH="next_query_point_growing_2d.npy",
      #  MINIMISE=True
    #)

   # propose_next_query_point_growing_Dd_tuned(
     #   D=3, X_PATH=baseX, Y_PATH=baseY,
       # SAVE_PATH="next_query_point_growing_3d.npy",
       # MINIMISE=True
    #)

   # propose_next_query_point_growing_Dd_tuned(
     #   D=4, X_PATH=baseX, Y_PATH=baseY,
     #   SAVE_PATH="next_query_point_growing_4d.npy",
     #   MINIMISE=True
   # )

   # propose_next_query_point_growing_Dd_tuned(
     #   D=5, X_PATH=baseX, Y_PATH=baseY,
     #   SAVE_PATH="next_query_point_growing_5d.npy",
      #  MINIMISE=True
    #)

    #propose_next_query_point_growing_Dd_tuned(
       # D=6, X_PATH=baseX, Y_PATH=baseY,
      #  SAVE_PATH="next_query_point_growing_6d.npy",
      #  MINIMISE=True
    #)

    propose_next_query_point_growing_Dd_tuned(
        D=8, X_PATH=baseX, Y_PATH=baseY,
        SAVE_PATH="next_query_point_growing_8d.npy",
        MINIMISE=True
    )


/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


=== BO Next Query (GROWING 8D tuned) ===
X_PATH: initial_inputs.npy
Y_PATH: initial_outputs.npy
n points: 40
Chosen XI: 0.001
Proposed next query point: [0.98594639 0.97398079 0.9988865  0.90298677 0.00964788 0.02211241
 0.99291549 0.9887561 ]
Saved to: next_query_point_growing_8d.npy
Current best y: 5.5921933895401965
Bounds: [(0.009075976680729044, 0.9859463896331099), (0.003418499484064614, 0.9739807874919709), (0.02292767798954412, 0.998886496855836), (0.009042485399991336, 0.9029867747482919), (0.009647877991745853, 0.986902999632929), (0.022112406876058893, 0.9902448143772585), (0.035907876169543856, 0.992915494924961), (0.041955069457023236, 0.9887561042522277)]
GP kernel (fit): 4.68**2 * RBF(length_scale=[1.89, 2.61, 1.34, 10.5, 12, 2.46, 1.9, 18.5]) + WhiteKernel(noise_level=1e-08)
